# Sessions &mdash; `save_session()` / `load_session()`

A **session** is a reloadable snapshot of a notebook's plotting state: every set's data
source, query, select flag and per-set formatting, plus the notebook-level formatting
(plot style, color/marker maps, default format, labels, lines, highlights, axis limits,
font sizes, ...). It is written as one JSON file and restored into any notebook later.

### What goes in the file

Each set is stored in one of two ways:

| Stored as | When | On reload |
|---|---|---|
| **file reference** | the set came from a file that still exists and its columns are unchanged since loading | the file is **re-read**, so later edits to the file show up |
| **embedded data** | in-memory loads, derived sets (`delta`/`combine`), df-replaced sets, sets with added or overwritten columns, or a missing source file | the rows come back **exactly** as saved (float64 round-trips bit-for-bit) |

The `embed_data` argument controls the policy:

- `True` (default) &mdash; embed only what cannot be reproduced from a file.
- `'all'` &mdash; embed everything: a fully self-contained snapshot.
- `False` &mdash; file references only; sets without a file source are skipped with a warning.

This notebook walks through each behaviour with assertions you can rerun.

## 0. Setup &mdash; a data file plus an in-memory frame

Everything below happens in a throw-away temporary folder so the repo stays clean. We write
one CSV holding two engine variants split by `SETNUMBER`, and build a third variant purely
in memory (the kind of data that has no file to point back to).

In [1]:
# --- make repo-root importable (notebook lives in demo_notebooks/) ---
import sys, os
_repo_root = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

import json, shutil, tempfile
from pathlib import Path
import numpy as np
import pandas as pd
from unichart import UnichartNotebook

work = Path(tempfile.mkdtemp(prefix='unichart_session_demo_'))
data_dir = work / 'data'
data_dir.mkdir()

n1 = np.array([40, 55, 70, 85, 100], float)
def engine(setnum, title, k_ff, k_egt):
    return pd.DataFrame({'SETNUMBER': setnum, 'TITLE': title, 'N1': n1,
                         'FF': k_ff * n1 ** 2 / 10, 'EGT': 300 + k_egt * n1})

csv_path = data_dir / 'engines.csv'
pd.concat([engine(0, 'Engine A', 4.8, 4.6), engine(1, 'Engine B', 5.2, 4.9)],
          ignore_index=True).to_csv(csv_path, index=False)

mem_df = engine(2, 'Engine C (in-memory)', 4.4, 4.3).drop(columns=['SETNUMBER', 'TITLE'])
print('work folder:', work)

work folder: /tmp/unichart_session_demo_k9w7k0of


## 1. Build a session worth keeping

Load the file and the in-memory frame, then apply a mix of per-set and notebook-level
formatting so there is something to verify after the round-trip.

In [2]:
uc = UnichartNotebook()
uc.load(csv_path)                                 # two file-backed sets (split on SETNUMBER)
uc.load_df(mem_df, title='Engine C (in-memory)')  # one memory-only set

# Per-set formatting
uc.color(0, '#1f77b4'); uc.color(1, '#d62728'); uc.color(2, '#2ca02c')
uc.marker(1, 's')
uc.query(2, 'N1 >= 55')          # saved as an expression and re-run on load
uc.sets[1].zorder = 5

# Notebook-level formatting
uc.set_plot_style('plotly')
uc.line('N1', 70, color='gray', label='design point')
uc.highlight('EGT', (700, 800), color='orange', alpha=0.15)
uc.scale('FF', (0, 6000))
uc.suptitle = 'Engine sweep - saved session'

uc.plot('N1', ['FF', 'EGT'])

UniChart Notebook Environment Initialized.
Loaded Set 0: Engine A
Loaded Set 1: Engine B
Loaded Set 2: Engine C (in-memory)
Plot style set to: plotly (existing datasets restyled)
Limits set for 'FF': (0, 6000)


## 2. Save

The summary line reports how many sets were file-referenced versus embedded. The two
sets from `engines.csv` become references; the in-memory set must be embedded.

In [3]:
session_path = work / 'engines_session.json'
uc.save_session(session_path)

session = json.loads(session_path.read_text())
print('\nformat version :', session['unichart_session'])
print('top-level keys :', list(session))
for i, entry in enumerate(session['sets']):
    src = entry['source']
    detail = src['rel_path'] if src['kind'] == 'file' else f"{len(src['frame']['data']['N1'])} rows"
    print(f"set {i}: {entry['title']!r:24} -> {src['kind']:8} ({detail})")
print('file size      :', f'{session_path.stat().st_size / 1024:.1f} kB')

Saved session to /tmp/unichart_session_demo_k9w7k0of/engines_session.json: 3 set(s) (2 file-referenced, 1 embedded)

format version : 1
top-level keys : ['unichart_session', 'saved', 'notebook', 'sets']
set 0: 'Engine A'               -> file     (data/engines.csv)
set 1: 'Engine B'               -> file     (data/engines.csv)
set 2: 'Engine C (in-memory)'   -> embedded (5 rows)
file size      : 4.9 kB


## 3. Reload into a fresh notebook

`load_session` returns the created datasets. Data, queries, per-set formatting and the
notebook decorations all come back; the plot below should be identical to the one above.

In [4]:
uc2 = UnichartNotebook()
restored = uc2.load_session(session_path)

for a, b in zip(uc.sets, uc2.sets):
    assert a.title == b.title
    assert a.df.equals(b.df), a.title                     # query applied identically
    assert (a.color, a.marker, a.query, a.zorder) == (b.color, b.marker, b.query, b.zorder)
assert len(uc2.sets[2].df) == 4, 'query re-applied on load'
assert uc2.plot_style == 'plotly' and uc2.suptitle == uc.suptitle
# JSON turns tuples into lists (e.g. the (0, 6000) limit), which the plotters accept.
norm = lambda o: json.loads(json.dumps(o))
assert norm(uc2.lines) == norm(uc.lines) and norm(uc2.axis_limits) == norm(uc.axis_limits)
assert uc2.sets[0].file_path == str(csv_path), 'file-backed sets keep their provenance'
print('PASS - data, queries, formatting and decorations all restored.')

uc2.plot('N1', ['FF', 'EGT'])

UniChart Notebook Environment Initialized.
Loaded session from /tmp/unichart_session_demo_k9w7k0of/engines_session.json: 3 set(s) restored
PASS - data, queries, formatting and decorations all restored.


## 4. Edits that silently switch a set to *embedded*

A file reference is only used when the file can still reproduce the set. Anything that
changes the set's columns or values through the write APIs is detected, and the set is
embedded instead:

- adding a derived column (`ds['NEW'] = ...`, `set_column`, `add_column`)
- **overwriting** a column that came from the file
- replacing the rows outright (`ds.df = ...`)
- derived sets from `delta()` / `combine()`, which never had a file

Below, Engine A gains a derived column and Engine B has `FF` overwritten. Both would be
wrong if reloaded from the CSV, so `save_session` embeds them.

In [5]:
uc.sets[0]['FF_PER_N1'] = uc.sets[0]['FF'] / uc.sets[0]['N1']    # derived column
uc.set_column(1, 'FF', uc.sets[1]['FF'] * 1.05)                     # overwrite loaded values

edited_path = work / 'edited_session.json'
uc.save_session(edited_path)

kinds = [e['source']['kind'] for e in json.loads(edited_path.read_text())['sets']]
print('stored as:', kinds)
assert kinds == ['embedded', 'embedded', 'embedded']

uc3 = UnichartNotebook()
uc3.load_session(edited_path)
assert 'FF_PER_N1' in uc3.sets[0].columns
assert np.allclose(uc3.sets[1]['FF'], uc.sets[1]['FF']), 'overwritten values must survive'
print('PASS - derived and overwritten columns survive the round-trip.')

Saved session to /tmp/unichart_session_demo_k9w7k0of/edited_session.json: 3 set(s) (0 file-referenced, 3 embedded)
stored as: ['embedded', 'embedded', 'embedded']
UniChart Notebook Environment Initialized.
Loaded session from /tmp/unichart_session_demo_k9w7k0of/edited_session.json: 3 set(s) restored
PASS - derived and overwritten columns survive the round-trip.


## 5. `embed_data='all'` &mdash; a snapshot that ignores later file edits

File references *follow* the file: if `engines.csv` changes after saving, a reload picks up
the change. That is usually what you want for a living dataset. When you need an archival
snapshot instead, pass `embed_data='all'`.

Here we save both flavours, then modify the CSV on disk and reload each.

In [6]:
uc = UnichartNotebook()
uc.load(csv_path)
ref_path  = work / 'reference_session.json'
snap_path = work / 'snapshot_session.json'
uc.save_session(ref_path)                      # file references
uc.save_session(snap_path, embed_data='all')   # everything embedded

# Now scale EGT in the CSV on disk.
edited = pd.read_csv(csv_path)
edited['EGT'] = edited['EGT'] * 2
edited.to_csv(csv_path, index=False)

from_ref  = UnichartNotebook(); from_ref.load_session(ref_path)
from_snap = UnichartNotebook(); from_snap.load_session(snap_path)

print('\nEGT at N1=40, Engine A:')
print('  original     :', uc.sets[0]["EGT"].iloc[0])
print('  file-ref load:', from_ref.sets[0]["EGT"].iloc[0], '  <- follows the edited file')
print('  snapshot load:', from_snap.sets[0]["EGT"].iloc[0], '  <- frozen at save time')
assert from_ref.sets[0]['EGT'].iloc[0] == 2 * uc.sets[0]['EGT'].iloc[0]
assert from_snap.sets[0]['EGT'].iloc[0] == uc.sets[0]['EGT'].iloc[0]
print('PASS')

UniChart Notebook Environment Initialized.
Loaded Set 0: Engine A
Loaded Set 1: Engine B
Saved session to /tmp/unichart_session_demo_k9w7k0of/reference_session.json: 2 set(s) (2 file-referenced, 0 embedded)
Saved session to /tmp/unichart_session_demo_k9w7k0of/snapshot_session.json: 2 set(s) (0 file-referenced, 2 embedded)
UniChart Notebook Environment Initialized.
Loaded session from /tmp/unichart_session_demo_k9w7k0of/reference_session.json: 2 set(s) restored
UniChart Notebook Environment Initialized.
Loaded session from /tmp/unichart_session_demo_k9w7k0of/snapshot_session.json: 2 set(s) restored

EGT at N1=40, Engine A:
  original     : 484.0
  file-ref load: 968.0   <- follows the edited file
  snapshot load: 484.0   <- frozen at save time
PASS


## 6. `embed_data=False` &mdash; references only

With `embed_data=False` nothing is embedded. Sets that cannot be reproduced from a file are
skipped, and the summary line says which ones.

In [7]:
uc.load_df(mem_df, title='Engine C (in-memory)')
refs_only = work / 'refs_only_session.json'
uc.save_session(refs_only, embed_data=False)
n_saved = len(json.loads(refs_only.read_text())['sets'])
assert n_saved == 2, n_saved
print('PASS - the in-memory set was skipped, two file-backed sets saved.')

Loaded Set 2: Engine C (in-memory)
Saved session to /tmp/unichart_session_demo_k9w7k0of/refs_only_session.json: 2 set(s) (2 file-referenced, 0 embedded); skipped 1 set(s) with no file source [2] (use embed_data=True to include them)
PASS - the in-memory set was skipped, two file-backed sets saved.


## 7. Moving a session with its data

File references store both the absolute path and the path **relative to the session
file**. If the absolute path no longer exists, the relative one is tried, so a project
folder can be moved, zipped or shared as a unit.

In [8]:
moved = work.parent / (work.name + '_moved')
shutil.move(str(work), str(moved))
work = moved                                    # keep later cells pointing at the moved folder
csv_path = work / 'data' / 'engines.csv'

uc4 = UnichartNotebook()
uc4.load_session(work / 'reference_session.json')
assert len(uc4.sets) == 2
assert uc4.sets[0].file_path == str(work / 'data' / 'engines.csv'), uc4.sets[0].file_path
print('PASS - resolved via the relative path; provenance now points at the moved file.')

UniChart Notebook Environment Initialized.
Loaded session from /tmp/unichart_session_demo_k9w7k0of_moved/reference_session.json: 2 set(s) restored
PASS - resolved via the relative path; provenance now points at the moved file.


## 8. Appending and keeping your own notebook formatting

`load_session` **appends** to whatever is already loaded (set indices shift accordingly,
and `delta_sets` base/study references are remapped). Pass
`restore_notebook_format=False` to bring in the sets only and keep the current notebook's
style, maps and decorations.

In [9]:
uc5 = UnichartNotebook()
uc5.load_df(mem_df, title='Already here')
uc5.set_plot_style('matplotlib')
uc5.load_session(work / 'reference_session.json', restore_notebook_format=False)

assert [d.index for d in uc5.sets] == [0, 1, 2]
assert uc5.plot_style == 'matplotlib', 'notebook formatting untouched'
print('PASS -', [d.title for d in uc5.sets])

UniChart Notebook Environment Initialized.
Loaded Set 0: Already here
Plot style set to: matplotlib (existing datasets restyled)
Loaded session from /tmp/unichart_session_demo_k9w7k0of_moved/reference_session.json: 2 set(s) restored (appended after 1 existing)
PASS - ['Already here', 'Engine A', 'Engine B']


## 9. The one blind spot

Writes through `ds[col] = ...`, `set_column` and `add_column` are tracked. Editing the
combined frame **in place** through `nb.df` is not &mdash; there is no hook to notice it &mdash; so
a file-backed set edited that way would reload from the file with the edit lost.

If you edit `nb.df` directly, save with `embed_data='all'`.

In [10]:
uc6 = UnichartNotebook()
uc6.load(csv_path)
uc6.df.loc[uc6.df.index[0], 'FF'] = -1.0          # in-place edit, not tracked

blind = work / 'blind_spot.json'
uc6.save_session(blind)                            # still a file reference...
check = UnichartNotebook(); check.load_session(blind)
print('\nreloaded FF[0]:', check.sets[0]['FF'].iloc[0], ' (the -1.0 edit is gone)')

uc6.save_session(blind, embed_data='all')          # ...unless you snapshot
check = UnichartNotebook(); check.load_session(blind)
assert check.sets[0]['FF'].iloc[0] == -1.0
print('reloaded FF[0]:', check.sets[0]['FF'].iloc[0], " (kept with embed_data='all')")

UniChart Notebook Environment Initialized.
Loaded Set 0: Engine A
Loaded Set 1: Engine B
Saved session to /tmp/unichart_session_demo_k9w7k0of_moved/blind_spot.json: 2 set(s) (2 file-referenced, 0 embedded)
UniChart Notebook Environment Initialized.
Loaded session from /tmp/unichart_session_demo_k9w7k0of_moved/blind_spot.json: 2 set(s) restored

reloaded FF[0]: 768.0  (the -1.0 edit is gone)
Saved session to /tmp/unichart_session_demo_k9w7k0of_moved/blind_spot.json: 2 set(s) (0 file-referenced, 2 embedded)
UniChart Notebook Environment Initialized.
Loaded session from /tmp/unichart_session_demo_k9w7k0of_moved/blind_spot.json: 2 set(s) restored
reloaded FF[0]: -1.0  (kept with embed_data='all')


## 10. Cleanup

In [11]:
shutil.rmtree(work, ignore_errors=True)
print('removed', work)

removed /tmp/unichart_session_demo_k9w7k0of_moved
